# Orders Bronze — Production ingestion

Supports `INCREMENTAL`, `BACKFILL`, and `FULL_LOAD` for `orders_YYYYMMDD.json` files.

- Explicit source schema is applied to every JSON read.
- Shared global manifest at `metadata/file_manifest/`; filters strictly to `orders` records.
- `INCREMENTAL` skips successful files unless content changed; `BACKFILL` uses date range; `FULL_LOAD` includes successful files.
- Processing writes directly to the dataset's Bronze Delta path. There is no dry-run toggle.
- Use one writer per dataset. Bronze and manifest updates are separate Delta transactions; reconcile if finalization fails after a Bronze write.


In [0]:
# CELL 1 — CONFIGURATION, EXPLICIT SCHEMA, AND WIDGETS
from delta.tables import DeltaTable
from pyspark.sql import functions as F                 
from pyspark.sql.types import StructType, StructField, LongType, StringType, DoubleType
from datetime import datetime, timezone, timedelta
import uuid

STORAGE_ACCOUNT = "nestlesalesdatalake"
CONTAINER = "customer-sales-datalake"
BASE_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
DATASET_NAME = "orders"
SOURCE_SYSTEM = "orders_api"
LANDING_PATH = BASE_PATH + f"landing/{DATASET_NAME}/"
BRONZE_PATH = BASE_PATH + f"bronze/{DATASET_NAME}/"
MANIFEST_PATH = BASE_PATH + "metadata/file_manifest/"
LEASE_MINUTES = 10
SOURCE_SCHEMA = StructType([
    StructField("customer_id", LongType(), False),
    StructField("market", StringType(), True),
    StructField("order_id", LongType(), False),
    StructField("order_timestamp", StringType(), True),
    StructField("product_id", LongType(), False),
    StructField("quantity", LongType(), False),
    StructField("sales_amount", StringType(), True),  # JSON stores this as text
    StructField("updated_at", StringType(), True)
])

BRONZE_COLUMNS = ['customer_id', 'market', 'order_id', 'order_timestamp', 'product_id', 'quantity', 'sales_amount', 'updated_at', 'ingestion_timestamp', 'source_system', 'source_file', 'run_id', 'batch_id']

# for name in ["run_mode", "backfill_start_date", "backfill_end_date", "reprocess_successful"]:
#     try:
#         dbutils.widgets.remove(name)
#     except Exception:
#         pass

dbutils.widgets.dropdown("run_mode", "INCREMENTAL", ["INCREMENTAL", "BACKFILL", "FULL_LOAD"], "Run Mode")
dbutils.widgets.text("backfill_start_date", "", "Backfill Start Date (YYYY-MM-DD)")
dbutils.widgets.text("backfill_end_date", "", "Backfill End Date (YYYY-MM-DD)")
dbutils.widgets.dropdown("reprocess_successful", "false", ["false", "true"], "Reprocess Successful Files?")

RUN_MODE = dbutils.widgets.get("run_mode").strip().upper()
BACKFILL_START_DATE = dbutils.widgets.get("backfill_start_date").strip()
BACKFILL_END_DATE = dbutils.widgets.get("backfill_end_date").strip()
REPROCESS_SUCCESSFUL = dbutils.widgets.get("reprocess_successful").strip().lower() == "true"

assert RUN_MODE in {"INCREMENTAL", "BACKFILL", "FULL_LOAD"} 
if RUN_MODE == "BACKFILL":
    assert BACKFILL_START_DATE and BACKFILL_END_DATE, "BACKFILL requires both dates."
    start_date = datetime.strptime(BACKFILL_START_DATE, "%Y-%m-%d").date()
    end_date = datetime.strptime(BACKFILL_END_DATE, "%Y-%m-%d").date()
    assert start_date <= end_date, "Backfill start date must be <= end date."

print("Dataset:", DATASET_NAME, "| Mode:", RUN_MODE, "| Reprocess successful:", REPROCESS_SUCCESSFUL)
print("Landing:", LANDING_PATH)
print("Bronze:", BRONZE_PATH)
print("Manifest:", MANIFEST_PATH)


Dataset: orders | Mode: INCREMENTAL | Reprocess successful: True
Landing: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/
Bronze: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/orders/
Manifest: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/metadata/file_manifest/


In [0]:


# CELL 2 — DISCOVER LANDING FILES AND HASH CONTENT (READ-ONLY)

def list_files_recursive(path):
    found = []

    for item in dbutils.fs.ls(path):
        if item.isDir():
            found.extend(list_files_recursive(item.path))
        else:
            found.append((item.name, item.path, int(item.size)))

    return found


# 1. Discover landing files
discovered = list_files_recursive(LANDING_PATH)

landing_schema = StructType([
    StructField("file_name", StringType(), False),
    StructField("file_path", StringType(), False),
    StructField("file_size_bytes", LongType(), False),
])

landing_df = (
    spark.createDataFrame(discovered, landing_schema)
    if discovered
    else spark.createDataFrame([], landing_schema)
)


# 2. Extract and validate dates from filenames
# Expected format: orders_YYYYMMDD.json
dated_df = (
    landing_df
    .withColumn(
        "date_string",
        F.regexp_extract(
            F.col("file_name"),
            r"^orders_(\d{8})\.json$",
            1
        )
    )
    .withColumn(
        "data_date",
        F.expr("try_to_date(date_string, 'yyyyMMdd')")
    )
    .withColumn(
        "filename_status",
        F.when(
            F.col("date_string") == "",
            F.lit("INVALID_FILENAME")
        )
        .when(
            F.col("data_date").isNull(),
            F.lit("INVALID_DATE")
        )
        .otherwise(F.lit("VALID"))
    )
)

# Show invalid files before stopping
invalid_df = dated_df.filter(F.col("filename_status") != "VALID")

if invalid_df.limit(1).count() > 0:
    display(
        invalid_df.select(
            "file_name",
            "file_path",
            "date_string",
            "filename_status"
        )
    )
    raise ValueError(
        "Invalid Orders filenames or dates found. "
        "Review the displayed files before processing."
    )


# 3. Calculate SHA-256 hashes
binary_df = (
    spark.read.format("binaryFile")
    .load(LANDING_PATH + "**")
    .select(
        F.col("path").alias("file_path"),
        F.sha2("content", 256).alias("actual_content_hash")
    )
)


# 4. Join file metadata with hashes
files_df = (
    dated_df
    .filter(F.col("filename_status") == "VALID")
    .join(binary_df, on="file_path", how="left")
)

# Ensure every discovered file was hashed
unhashed_df = files_df.filter(
    F.col("actual_content_hash").isNull()
)

if unhashed_df.limit(1).count() > 0:
    display(unhashed_df.select("file_name", "file_path"))
    raise ValueError(
        "Could not hash every discovered Orders file."
    )


# 5. Display discovery results
display(
    files_df.select(
        "file_name",
        "file_path",
        "data_date",
        "file_size_bytes",
        "actual_content_hash"
    ).orderBy("data_date", "file_name")
)

print("Discovered Orders files:", files_df.count())


file_name,file_path,data_date,file_size_bytes,actual_content_hash
orders_20261005.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=05/orders_20261005.json,2026-10-05,4859546,3f321bd41cf7ae641985e249e621016f138966ce6527331d2b1224d1caa3c9b4
orders_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=07/orders_20261007.json,2026-10-07,6073246,19d8de3c2b51f519b3528fd93aa15766dec2a0ece904609880c606b20ee682af
orders_20261008.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=08/orders_20261008.json,2026-10-08,4712847,4a9412127a3e037d1afd64c27dd32d934b91351d607bcb20c9d710edfef92d3d
orders_20261010.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=10/orders_20261010.json,2026-10-10,4884518,29a7e1224f4005946fbcb930ea0ed7eca3c5b1755b89c605fa63d584de226c0b
orders_20261012.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=12/orders_20261012.json,2026-10-12,731,f1ba2e7f95db79cbb7a3c88e7d15d45d40b7825d25f70279aa35b7bad212c509


Discovered Orders files: 5


In [0]:
# CELL 3 — MANIFEST RECONCILIATION AND RUN-MODE SELECTION (READ-ONLY)
if not DeltaTable.isDeltaTable(spark, MANIFEST_PATH):
    raise RuntimeError(f"Global manifest Delta table not found: {MANIFEST_PATH}")

manifest_all_df = spark.read.format("delta").load(MANIFEST_PATH)
required_manifest_columns = {
    "dataset", "file_name", "file_path", "status", "content_hash",
    "data_date", "batch_id", "run_id", "retry_count", "attempt_count",
    "claimed_at", "lease_expires_at", "last_attempt_at", "status_updated_at",
    "error_message", "error_type", "last_error_at", "processed_at",
    "last_success_at", "processed_content_hash", "source_record_count",
    "written_record_count", "rejected_record_count", "target_path",
    "target_delta_version", "file_size_bytes", "first_seen", "last_seen_at",
    "discovery_run_id"
}
missing = sorted(required_manifest_columns - set(manifest_all_df.columns))
if missing:
    raise ValueError(f"Global manifest missing required columns: {missing}")

manifest_df = (
    manifest_all_df.filter(F.col("dataset") == DATASET_NAME)
    .select("dataset", "file_name", "file_path", "status", "content_hash",
            "data_date", "batch_id", "run_id", "retry_count", "attempt_count")
)
duplicate_keys = manifest_df.groupBy("dataset", "file_name").count().filter("count > 1").count()
assert duplicate_keys == 0, "Duplicate dataset/file_name keys in manifest."

reconciliation_df = (
    files_df.alias("l")
    .join(manifest_df.alias("m"), F.col("l.file_name") == F.col("m.file_name"), "left")
    .select(
        F.col("l.file_name"), F.col("l.file_path"), F.col("l.file_size_bytes"),
        F.col("l.data_date").alias("landing_data_date"), F.col("l.actual_content_hash"),
        F.col("m.status").alias("manifest_status"), F.col("m.content_hash").alias("manifest_hash"),
        F.col("m.data_date").alias("manifest_data_date"), F.col("m.file_path").alias("manifest_file_path")
    )
    .withColumn("effective_data_date", F.coalesce("manifest_data_date", "landing_data_date"))
    .withColumn(
        "selection_reason",
        F.when(F.col("manifest_status") == "PROCESSING", "SKIP_PROCESSING")
         .when(F.col("manifest_status").isNull(), "REGISTER_REQUIRED")
         .when(F.col("manifest_hash").isNull(), "REVIEW_REQUIRED")
         .when(F.col("manifest_hash") != F.col("actual_content_hash"), "CONTENT_CHANGED")
         .when(F.col("manifest_status").isin("NEW", "FAILED"), "ELIGIBLE")
         .when(F.col("manifest_status") == "SUCCESS", "SUCCESS_REPROCESS_CANDIDATE")
         .otherwise("REVIEW_REQUIRED")
    )
)

base_reasons = ["REGISTER_REQUIRED", "ELIGIBLE", "CONTENT_CHANGED"]
if RUN_MODE == "INCREMENTAL":
    selected_df = reconciliation_df.filter(F.col("selection_reason").isin(base_reasons))
elif RUN_MODE == "BACKFILL":
    reasons = base_reasons + (["SUCCESS_REPROCESS_CANDIDATE"] if REPROCESS_SUCCESSFUL else [])
    selected_df = reconciliation_df.filter(
        F.col("effective_data_date").between(
            F.to_date(F.lit(BACKFILL_START_DATE)), F.to_date(F.lit(BACKFILL_END_DATE))
        ) & F.col("selection_reason").isin(reasons)
    )
else:
    selected_df = reconciliation_df.filter(
        F.col("selection_reason").isin(base_reasons + ["SUCCESS_REPROCESS_CANDIDATE"])
    )

display(reconciliation_df.select("file_name", "effective_data_date", "manifest_status",
                                 "selection_reason").orderBy("effective_data_date", "file_name"))
print("Selected candidates:", selected_df.count())


file_name,effective_data_date,manifest_status,selection_reason
orders_20261005.json,2026-10-05,SUCCESS,SUCCESS_REPROCESS_CANDIDATE
orders_20261007.json,2026-10-07,SUCCESS,SUCCESS_REPROCESS_CANDIDATE
orders_20261008.json,2026-10-08,SUCCESS,SUCCESS_REPROCESS_CANDIDATE
orders_20261010.json,2026-10-10,SUCCESS,SUCCESS_REPROCESS_CANDIDATE
orders_20261012.json,2026-10-12,SUCCESS,SUCCESS_REPROCESS_CANDIDATE


Selected candidates: 0


In [0]:
# CELL 4 — PREFLIGHT AND SCHEMA CHECKS (READ-ONLY)
selected_count = selected_df.count()
assert selected_df.groupBy("file_name").count().filter("count > 1").count() == 0, "Duplicate selected files."
assert selected_df.filter(F.col("actual_content_hash").isNull() | (F.length("actual_content_hash") != 64)).count() == 0, "Missing/invalid source hash."
assert selected_df.filter(F.col("effective_data_date").isNull()).count() == 0, "Missing data date."
assert selected_df.filter(F.col("manifest_status") == "PROCESSING").count() == 0, "A PROCESSING file was selected."
assert selected_df.filter(F.col("selection_reason") == "REVIEW_REQUIRED").count() == 0, "A review-required file was selected."

if DeltaTable.isDeltaTable(spark, BRONZE_PATH):
    existing_bronze_df = spark.read.format("delta").load(BRONZE_PATH)
    missing_bronze = sorted(set(BRONZE_COLUMNS) - set(existing_bronze_df.columns))
    extra_bronze = sorted(set(existing_bronze_df.columns) - set(BRONZE_COLUMNS))
    if missing_bronze or extra_bronze:
        raise ValueError(f"Existing Bronze columns mismatch. Missing={missing_bronze}; extra={extra_bronze}")
else:
    print("Bronze Delta target does not exist yet; it will be created on first write.")

display(selected_df.select("file_name", "file_path", "effective_data_date",
                           "manifest_status", "selection_reason", "actual_content_hash"))
print("Preflight PASS | selected:", selected_count)


file_name,file_path,effective_data_date,manifest_status,selection_reason,actual_content_hash


Preflight PASS | selected: 0


In [0]:
# CELL 5 — PROCESS SELECTED FILES
# This cell performs manifest and Bronze writes. Use ONE writer for this dataset.
run_id = str(uuid.uuid4())
manifest_delta = DeltaTable.forPath(spark, MANIFEST_PATH)
manifest_schema = spark.read.format("delta").load(MANIFEST_PATH).schema
manifest_columns = [f.name for f in manifest_schema.fields]
results = []
print("Processing run ID:", run_id)

def register_manifest_file(file_name, file_path, file_size, data_date, content_hash):
    now = datetime.now(timezone.utc).replace(tzinfo=None)
    values = {name: None for name in manifest_columns}
    values.update({
        "dataset": DATASET_NAME, "file_name": file_name, "file_path": file_path,
        "status": "NEW", "first_seen": now, "batch_id": str(uuid.uuid4()), "retry_count": 0,
        "data_date": data_date, "file_size_bytes": int(file_size),
        "content_hash": content_hash, "last_seen_at": now,
        "discovery_run_id": run_id, "attempt_count": 0, "status_updated_at": now
    })
    new_row = spark.createDataFrame([values], schema=manifest_schema)
    (manifest_delta.alias("t")
        .merge(new_row.alias("s"), "t.dataset = s.dataset AND t.file_name = s.file_name")
        .whenNotMatchedInsertAll().execute())

for c in selected_df.orderBy("effective_data_date", "file_name").collect():
    file_name, file_path = c["file_name"], c["file_path"]
    actual_hash, reason = c["actual_content_hash"], c["selection_reason"]
    bronze_committed = False
    claimed = False
    try:
        current_hash = spark.read.format("binaryFile").load(file_path).select(
            F.sha2("content", 256).alias("h")
        ).first()["h"]
        assert current_hash == actual_hash, f"{file_name}: source changed after discovery."

        rows = (spark.read.format("delta").load(MANIFEST_PATH)
            .filter((F.col("dataset") == DATASET_NAME) & (F.col("file_name") == file_name))
            .select("status", "content_hash", "file_path").collect())

        if not rows:
            if reason != "REGISTER_REQUIRED":
                raise RuntimeError(f"{file_name}: manifest row disappeared; rerun discovery.")
            register_manifest_file(file_name, file_path, c["file_size_bytes"],
                                   c["effective_data_date"], actual_hash)
            rows = (spark.read.format("delta").load(MANIFEST_PATH)
                .filter((F.col("dataset") == DATASET_NAME) & (F.col("file_name") == file_name))
                .select("status", "content_hash", "file_path").collect())

        if len(rows) != 1:
            raise RuntimeError(f"{file_name}: expected exactly one manifest row.")
        old = rows[0]
        if old["status"] == "PROCESSING":
            raise RuntimeError(f"{file_name}: already PROCESSING.")
        if old["file_path"] and old["file_path"] != file_path:
            raise RuntimeError(f"{file_name}: manifest file_path differs from discovered path.")
        if reason == "CONTENT_CHANGED":
            if old["content_hash"] == actual_hash:
                raise RuntimeError(f"{file_name}: content no longer appears changed; rerun discovery.")
        elif old["content_hash"] != actual_hash:
            raise RuntimeError(f"{file_name}: manifest hash mismatch; rerun discovery.")
        if old["status"] == "SUCCESS" and RUN_MODE == "INCREMENTAL" and reason != "CONTENT_CHANGED":
            raise RuntimeError(f"{file_name}: successful file not eligible for incremental reprocessing.")
        if old["status"] == "SUCCESS" and RUN_MODE == "BACKFILL" and not REPROCESS_SUCCESSFUL:
            raise RuntimeError(f"{file_name}: successful-file reprocessing disabled for this backfill.")

        expected_status, expected_hash = old["status"], old["content_hash"]
        now = datetime.now(timezone.utc).replace(tzinfo=None)
        lease_expiry = now + timedelta(minutes=LEASE_MINUTES)
        manifest_delta.update(
            condition=(
                (F.col("dataset") == DATASET_NAME) & (F.col("file_name") == file_name) &
                (F.col("status") == expected_status) & (F.col("content_hash") == expected_hash)
            ),
            set={
                "status": F.lit("PROCESSING"), "run_id": F.lit(run_id),
                "claimed_at": F.lit(now).cast("timestamp"),
                "lease_expires_at": F.lit(lease_expiry).cast("timestamp"),
                "last_attempt_at": F.lit(now).cast("timestamp"),
                "attempt_count": F.coalesce(F.col("attempt_count"), F.lit(0)) + 1,
                "status_updated_at": F.lit(now).cast("timestamp"),
                "content_hash": F.lit(actual_hash),
                "error_message": F.lit(None).cast("string"),
                "error_type": F.lit(None).cast("string")
            }
        )
        owner = (spark.read.format("delta").load(MANIFEST_PATH)
            .filter((F.col("dataset") == DATASET_NAME) & (F.col("file_name") == file_name))
            .select("status", "run_id", "content_hash").first())
        if not owner or owner["status"] != "PROCESSING" or owner["run_id"] != run_id or owner["content_hash"] != actual_hash:
            raise RuntimeError(f"{file_name}: manifest claim could not be verified.")
        claimed = True

        source_df = (spark.read.schema(SOURCE_SCHEMA)
            .option("multiLine", "true").option("mode", "FAILFAST").json(file_path))
        source_count = source_df.count()
        if source_count == 0:
            raise ValueError(f"{file_name}: source file is empty.")
        invalid_required = source_df.filter(
            F.col("customer_id").isNull() | F.col("order_id").isNull() |
            F.col("product_id").isNull() | F.col("quantity").isNull() |
            F.col("sales_amount").isNull()
        ).count()
        if invalid_required:
            raise ValueError(f"{file_name}: {invalid_required} rows have null required fields.")
        negative_quantity = source_df.filter(F.col("quantity") < 0).count()
        if negative_quantity:
            raise ValueError(f"{file_name}: negative quantity values found.")

        prepared_df = (
            source_df
        .withColumn("customer_id", F.col("customer_id").cast("long"))
        .withColumn("market", F.col("market").cast("string"))
        .withColumn("order_id", F.col("order_id").cast("long"))
        .withColumn("order_timestamp", F.col("order_timestamp").cast("string"))
        .withColumn("product_id", F.col("product_id").cast("long"))
        .withColumn("quantity", F.col("quantity").cast("long"))
        .withColumn("sales_amount", F.col("sales_amount").cast("double"))
        .withColumn("updated_at", F.col("updated_at").cast("string"))
            .withColumn("ingestion_timestamp", F.current_timestamp())
            .withColumn("source_system", F.lit(SOURCE_SYSTEM))
            .withColumn("source_file", F.lit(file_path))
            .withColumn("run_id", F.lit(run_id))
            .withColumn("batch_id", F.lit(
                spark.read.format("delta").load(MANIFEST_PATH)
                .filter((F.col("dataset") == DATASET_NAME) & (F.col("file_name") == file_name))
                .select("batch_id").first()["batch_id"]
            ))
            .select(*BRONZE_COLUMNS)
        )

        # Existing Bronze schema must match exactly before a write.
        if DeltaTable.isDeltaTable(spark, BRONZE_PATH):
            target_df = spark.read.format("delta").load(BRONZE_PATH)
            expected = [(f.name, f.dataType.simpleString()) for f in prepared_df.schema]
            actual = [(f.name, f.dataType.simpleString()) for f in target_df.schema]
            if expected != actual:
                raise RuntimeError(f"{file_name}: Bronze schema mismatch. Prepared={expected}; target={actual}")

        # Full-load/backfill/content-change replaces this source file only, preserving all other files.
        # Escape any single quote in the path for Delta replaceWhere SQL.
        path_literal = file_path.replace("'", "''")
        (prepared_df.write.format("delta").mode("overwrite")
            .option("replaceWhere", f"source_file = '{path_literal}'")
            .save(BRONZE_PATH))
        bronze_committed = True

        post_df = spark.read.format("delta").load(BRONZE_PATH).filter(F.col("source_file") == file_path)
        written_count = post_df.count()
        assert written_count == source_count, f"{file_name}: source/written count mismatch."
        assert post_df.filter(F.col("order_id").isNull()).count() == 0, f"{file_name}: null order_id after write."

        delta_version = DeltaTable.forPath(spark, BRONZE_PATH).history(1).select("version").first()["version"]
        finished = datetime.now(timezone.utc).replace(tzinfo=None)
        DeltaTable.forPath(spark, MANIFEST_PATH).alias("t").update(
            condition=(
                (F.col("t.dataset") == DATASET_NAME) & (F.col("t.file_name") == file_name) &
                (F.col("t.status") == "PROCESSING") & (F.col("t.run_id") == run_id) &
                (F.col("t.content_hash") == actual_hash)
            ),
            set={
                "status": F.lit("SUCCESS"), "processed_at": F.lit(finished).cast("timestamp"),
                "last_success_at": F.lit(finished).cast("timestamp"),
                "status_updated_at": F.lit(finished).cast("timestamp"),
                "processed_content_hash": F.lit(actual_hash),
                "source_record_count": F.lit(source_count).cast("long"),
                "written_record_count": F.lit(written_count).cast("long"),
                "rejected_record_count": F.lit(0).cast("long"),
                "target_path": F.lit(BRONZE_PATH),
                "target_delta_version": F.lit(delta_version).cast("long"),
                "error_message": F.lit(None).cast("string"),
                "error_type": F.lit(None).cast("string"),
                "last_error_at": F.lit(None).cast("timestamp"),
                "claimed_at": F.lit(None).cast("timestamp"),
                "lease_expires_at": F.lit(None).cast("timestamp")
            }
        )
        final = (spark.read.format("delta").load(MANIFEST_PATH)
            .filter((F.col("dataset") == DATASET_NAME) & (F.col("file_name") == file_name))
            .select("status", "run_id", "written_record_count").first())
        if not final or final["status"] != "SUCCESS" or final["run_id"] != run_id or final["written_record_count"] != written_count:
            raise RuntimeError(f"{file_name}: final manifest verification failed.")
        results.append((file_name, "SUCCESS", source_count, written_count, delta_version, ""))
        print(f"SUCCESS {file_name}: source={source_count}, written={written_count}, Delta version={delta_version}")

    except Exception as exc:
        error_text = f"{type(exc).__name__}: {str(exc)}"[:4000]
        if claimed:
            try:
                current_owner = (spark.read.format("delta").load(MANIFEST_PATH)
                    .filter((F.col("dataset") == DATASET_NAME) & (F.col("file_name") == file_name))
                    .select("status", "run_id").first())
                if current_owner and current_owner["status"] == "PROCESSING" and current_owner["run_id"] == run_id:
                    if bronze_committed:
                        error_text = ("BRONZE_WRITE_COMMITTED_BUT_FINALIZATION_FAILED; reconcile before retry. " + error_text)[:4000]
                    DeltaTable.forPath(spark, MANIFEST_PATH).alias("t").update(
                        condition=((F.col("t.dataset") == DATASET_NAME) &
                                   (F.col("t.file_name") == file_name) &
                                   (F.col("t.status") == "PROCESSING") &
                                   (F.col("t.run_id") == run_id)),
                        set={
                            "status": F.lit("FAILED"),
                            "status_updated_at": F.lit(datetime.now(timezone.utc).replace(tzinfo=None)).cast("timestamp"),
                            "last_error_at": F.current_timestamp(),
                            "error_message": F.lit(error_text),
                            "error_type": F.lit(type(exc).__name__),
                            "claimed_at": F.lit(None).cast("timestamp"),
                            "lease_expires_at": F.lit(None).cast("timestamp")
                        }
                    )
            except Exception as update_exc:
                print("Could not safely update manifest failure state:", str(update_exc)[:1000])
        status = "FAILED_AFTER_BRONZE_WRITE_RECONCILIATION_REQUIRED" if bronze_committed else "FAILED"
        print(f"{status} {file_name}: {error_text}")
        results.append((file_name, status, None, None, None, error_text))

results_df = spark.createDataFrame(
    results,
    "file_name string, status string, source_count long, written_count long, delta_version long, error_message string"
)
display(results_df)
print("Run ID:", run_id)


Processing run ID: 06151dd5-8f5d-48a4-9528-2117765d27a5


file_name,status,source_count,written_count,delta_version,error_message


Run ID: 06151dd5-8f5d-48a4-9528-2117765d27a5


In [0]:
# CELL 6 — FINAL BRONZE INVENTORY AND MANIFEST STATUS (READ-ONLY)
bronze_final_df = spark.read.format("delta").load(BRONZE_PATH)
manifest_final_df = spark.read.format("delta").load(MANIFEST_PATH)

display(
    bronze_final_df.filter(F.col("source_system") == SOURCE_SYSTEM)
    .groupBy("source_file")
    .agg(
        F.count("*").alias("bronze_rows"),
        F.countDistinct("order_id").alias("distinct_order_ids"),
        F.sum(F.when(F.col("order_id").isNull(), 1).otherwise(0)).alias("null_order_ids")
    )
    .orderBy("source_file")
)

display(
    manifest_final_df.filter(F.col("dataset") == DATASET_NAME)
    .select("file_name", "status", "run_id", "source_record_count",
            "written_record_count", "target_delta_version", "lease_expires_at", "error_message")
    .orderBy("file_name")
)
print("Total orders Bronze rows:", bronze_final_df.filter(F.col("source_system") == SOURCE_SYSTEM).count())


source_file,bronze_rows,distinct_order_ids,null_order_ids
abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=05/orders_20261005.json,25000,25000,0
abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=07/orders_20261007.json,25000,25000,0
abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=08/orders_20261008.json,25000,25000,0
abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=10/orders_20261010.json,25000,25000,0
abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=12/orders_20261012.json,3,3,0


file_name,status,run_id,source_record_count,written_record_count,target_delta_version,lease_expires_at,error_message
orders_20261005.json,SUCCESS,f3cb6e1d-55bf-48dc-b639-bc9bbfa8f9ea,25000,25000,9,null,null
orders_20261007.json,SUCCESS,f3cb6e1d-55bf-48dc-b639-bc9bbfa8f9ea,25000,25000,10,null,null
orders_20261008.json,SUCCESS,1211f387-4ccb-401c-bf62-037b611c410f,25000,25000,17,null,null
orders_20261010.json,SUCCESS,1211f387-4ccb-401c-bf62-037b611c410f,25000,25000,18,null,null
orders_20261012.json,SUCCESS,1211f387-4ccb-401c-bf62-037b611c410f,3,3,19,null,null


Total orders Bronze rows: 100003


In [0]:
# from delta.tables import DeltaTable
# from pyspark.sql import functions as F
# from datetime import datetime, timezone

# TARGET_DATASET = "orders"
# TARGET_FILE = "orders_20261008.json"

# manifest_delta = DeltaTable.forPath(spark, MANIFEST_PATH)

# # Verify exactly one matching record exists
# matches = (
#     spark.read.format("delta").load(MANIFEST_PATH)
#     .filter(
#         (F.col("dataset") == TARGET_DATASET) &
#         (F.col("file_name") == TARGET_FILE)
#     )
# )

# if matches.count() != 1:
#     raise RuntimeError(
#         f"Expected exactly one manifest record for {TARGET_FILE}."
#     )

# record = matches.select(
#     "status", "run_id", "lease_expires_at"
# ).first()

# if record["status"] != "PROCESSING":
#     raise RuntimeError(
#         f"Expected PROCESSING status, found {record['status']}. "
#         "No changes made."
#     )

# # Reset only this file, and only if its previous run still owns the record.
# now = datetime.now(timezone.utc).replace(tzinfo=None)

# manifest_delta.update(
#     condition=(
#         (F.col("dataset") == TARGET_DATASET) &
#         (F.col("file_name") == TARGET_FILE) &
#         (F.col("status") == "PROCESSING") &
#         (F.col("run_id") == record["run_id"]) &
#         (F.col("lease_expires_at") <= F.lit(now))
#     ),
#     set={
#         "status": F.lit("NEW"),
#         "run_id": F.lit(None).cast("string"),
#         "claimed_at": F.lit(None).cast("timestamp"),
#         "lease_expires_at": F.lit(None).cast("timestamp"),
#         "status_updated_at": F.lit(now).cast("timestamp"),
#         "error_message": F.lit(None).cast("string"),
#         "error_type": F.lit(None).cast("string")
#     }
# )

# # Verify the outcome
# display(
#     spark.read.format("delta").load(MANIFEST_PATH)
#     .filter(
#         (F.col("dataset") == TARGET_DATASET) &
#         (F.col("file_name") == TARGET_FILE)
#     )
#     .select(
#         "file_name", "status", "run_id",
#         "attempt_count", "retry_count",
#         "claimed_at", "lease_expires_at"
#     )
# )

In [0]:
# from pyspark.sql import functions as F

# display(
#     spark.read.format("delta").load(MANIFEST_PATH)
#     .filter(
#         (F.col("dataset") == "orders") &
#         (F.col("file_name") == "orders_20261008.json")
#     )
#     .select(
#         "file_name",
#         "status",
#         "run_id",
#         "claimed_at",
#         "lease_expires_at",
#         "last_attempt_at",
#         "attempt_count",
#         "error_message",
#         "file_path",
#         "content_hash"
#     )
# )

In [0]:
# from pyspark.sql import functions as F

# display(
#     spark.read.format("delta").load(MANIFEST_PATH)
#     .filter(
#         (F.col("dataset") == "orders") &
#         (F.col("file_name") == "orders_20261008.json")
#     )
#     .select(
#         "file_name",
#         "status",
#         "run_id",
#         "error_message",
#         "error_type",
#         "content_hash",
#          "processed_content_hash",
#         "attempt_count",
#         "retry_count"
#     )
# )

file_name,status,run_id,error_message,error_type,content_hash,processed_content_hash,attempt_count,retry_count
orders_20261008.json,SUCCESS,da442592-4664-4fb5-9a2b-be030aaec3f9,null,null,4a9412127a3e037d1afd64c27dd32d934b91351d607bcb20c9d710edfef92d3d,4a9412127a3e037d1afd64c27dd32d934b91351d607bcb20c9d710edfef92d3d,5,1


In [0]:

# from pyspark.sql import functions as F

# target_path = (
#     "abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/"
#     "landing/orders/year=2026/month=10/day=08/orders_20261008.json"
# )

# # Read a small sample without imposing the production schema
# sample_df = (
#     spark.read
#     .option("multiLine", "true")
#     .option("mode", "PERMISSIVE")
#     .json(target_path)
# )

# print("Inferred source schema:")
# sample_df.printSchema()

# display(sample_df.limit(10))


Inferred source schema:
root
 |-- customer_id: long (nullable = true)
 |-- market: string (nullable = true)
 |-- order_id: long (nullable = true)
 |-- order_timestamp: string (nullable = true)
 |-- product_id: long (nullable = true)
 |-- quantity: long (nullable = true)
 |-- sales_amount: string (nullable = true)
 |-- updated_at: string (nullable = true)



customer_id,market,order_id,order_timestamp,product_id,quantity,sales_amount,updated_at
190001,India,900001,2026-10-08T00:00:00,290001,1,80,2026-10-08T00:00:00
190002,Canada,900002,2026-10-08T00:00:47,290002,2,186,2026-10-08T00:01:47
190003,UK,900003,2026-10-08T00:01:34,290003,3,318,2026-10-08T00:03:34
190004,Australia,900004,2026-10-08T00:02:21,290004,4,476,2026-10-08T00:05:21
190005,Germany,900005,2026-10-08T00:03:08,290005,5,660,2026-10-08T00:07:08
190006,India,900006,2026-10-08T00:03:55,290006,6,870,2026-10-08T00:08:55
190007,Canada,900007,2026-10-08T00:04:42,290007,7,1106,2026-10-08T00:10:42
190008,UK,900008,2026-10-08T00:05:29,290008,8,1368,2026-10-08T00:12:29
190009,Australia,900009,2026-10-08T00:06:16,290009,1,184,2026-10-08T00:14:16
190010,Germany,900010,2026-10-08T00:07:03,290010,2,394,2026-10-08T00:16:03


In [0]:
# print("Current SOURCE_SCHEMA:")
# SOURCE_SCHEMA

Current SOURCE_SCHEMA:


StructType([StructField('customer_id', LongType(), False), StructField('market', StringType(), True), StructField('order_id', LongType(), False), StructField('order_timestamp', StringType(), True), StructField('product_id', LongType(), False), StructField('quantity', LongType(), False), StructField('sales_amount', StringType(), True), StructField('updated_at', StringType(), True)])

In [0]:

from pyspark.sql import functions as F

# Full ADLS Gen2 Delta table paths
STORAGE_ACCOUNT = "nestlesalesdatalake"
CONTAINER = "customer-sales-datalake"

BASE_PATH = (
    f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
)

datasets = [
    (
        "products",
        BASE_PATH + "bronze/products/",
        "product_id"
    ),
    (
        "customers",
        BASE_PATH + "bronze/customers/",
        "customer_id"
    ),
    (
        "orders",
        BASE_PATH + "bronze/orders/",
        "order_id"
    )
]

summary = []

for dataset_name, path, key_column in datasets:
    print(f"Validating {dataset_name}: {path}")

    df = spark.read.format("delta").load(path)

    stats = df.agg(
        F.count("*").alias("total_rows"),
        F.countDistinct(key_column).alias("distinct_keys"),
        F.sum(
            F.when(F.col(key_column).isNull(), 1).otherwise(0)
        ).alias("null_keys")
    ).first()

    summary.append((
        dataset_name,
        stats["total_rows"],
        stats["distinct_keys"],
        stats["null_keys"] or 0
    ))

summary_df = spark.createDataFrame(
    summary,
    """
    dataset string,
    total_rows long,
    distinct_keys long,
    null_keys long
    """
)

display(summary_df.orderBy("dataset"))

print(
    "Total Bronze rows:",
    summary_df.agg(F.sum("total_rows")).first()[0]
)


Validating products: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/products/
Validating customers: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/customers/
Validating orders: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/orders/


dataset,total_rows,distinct_keys,null_keys
customers,75000,74994,0
orders,50000,50000,0
products,50000,49994,0


Total Bronze rows: 175000


In [0]:

from pyspark.sql import functions as F

STORAGE_ACCOUNT = "nestlesalesdatalake"
CONTAINER = "customer-sales-datalake"

BASE_PATH = (
    f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
)

datasets = [
    ("customers", BASE_PATH + "bronze/customers/", "customer_id"),
    ("products", BASE_PATH + "bronze/products/", "product_id"),
]

for dataset_name, path, key_column in datasets:
    print(f"\n{'=' * 60}")
    print(f"Duplicate investigation: {dataset_name}")

    df = spark.read.format("delta").load(path)

    # Customers uses source_file; Products uses _source_file.
    source_col = (
        "source_file" if "source_file" in df.columns
        else "_source_file" if "_source_file" in df.columns
        else None
    )

    if source_col:
        duplicate_ids = (
            df.groupBy(key_column)
            .agg(
                F.count("*").alias("occurrences"),
                F.countDistinct(source_col).alias("source_file_count"),
                F.collect_set(source_col).alias("source_files")
            )
            .filter(F.col("occurrences") > 1)
            .orderBy(F.desc("occurrences"), key_column)
        )
    else:
        duplicate_ids = (
            df.groupBy(key_column)
            .agg(F.count("*").alias("occurrences"))
            .filter(F.col("occurrences") > 1)
            .orderBy(F.desc("occurrences"), key_column)
        )

    display(duplicate_ids)

    print(
        "Number of duplicated IDs:",
        duplicate_ids.count()
    )



Duplicate investigation: customers


customer_id,occurrences,source_file_count,source_files
130001,2,2,"List(abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json, abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=08/customers_20261008.json)"
130002,2,2,"List(abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=08/customers_20261008.json, abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json)"
130003,2,2,"List(abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json, abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=08/customers_20261008.json)"
130004,2,2,"List(abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json, abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=08/customers_20261008.json)"
130005,2,2,"List(abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json, abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=08/customers_20261008.json)"
130006,2,2,"List(abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json, abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=08/customers_20261008.json)"


Number of duplicated IDs: 6

Duplicate investigation: products


product_id,occurrences,source_file_count,source_files
230001,2,2,"List(products_20261007.json, products_20261008.json)"
230002,2,2,"List(products_20261007.json, products_20261008.json)"
230003,2,2,"List(products_20261007.json, products_20261008.json)"
230004,2,2,"List(products_20261007.json, products_20261008.json)"
230005,2,2,"List(products_20261007.json, products_20261008.json)"
230006,2,2,"List(products_20261007.json, products_20261008.json)"


Number of duplicated IDs: 6


In [0]:

from pyspark.sql import functions as F

BASE_PATH = (
    "abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/"
)

checks = [
    (
        "customers",
        BASE_PATH + "bronze/customers/",
        "customer_id",
        ["name", "city", "market"],
        "source_file"
    ),
    (
        "products",
        BASE_PATH + "bronze/products/",
        "product_id",
        ["product_name", "category", "brand", "price"],
        "_source_file"
    )
]

for dataset, path, key, attributes, source_col in checks:
    print(f"\n{dataset.upper()} — repeated record comparison")

    df = spark.read.format("delta").load(path)

    duplicates = (
        df.groupBy(key)
        .count()
        .filter(F.col("count") > 1)
        .select(key)
    )

    comparison = (
        df.join(duplicates, on=key, how="inner")
        .groupBy(key)
        .agg(
            F.count("*").alias("row_count"),
            F.countDistinct(
                F.struct(*[F.col(c) for c in attributes])
            ).alias("distinct_attribute_versions"),
            F.collect_set(source_col).alias("source_files")
        )
        .orderBy(key)
    )

    display(comparison)



CUSTOMERS — repeated record comparison


customer_id,row_count,distinct_attribute_versions,source_files
130001,2,2,"List(abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=08/customers_20261008.json, abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json)"
130002,2,2,"List(abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=08/customers_20261008.json, abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json)"
130003,2,2,"List(abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=08/customers_20261008.json, abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json)"
130004,2,2,"List(abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=08/customers_20261008.json, abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json)"
130005,2,2,"List(abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=08/customers_20261008.json, abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json)"
130006,2,2,"List(abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=08/customers_20261008.json, abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json)"



PRODUCTS — repeated record comparison


product_id,row_count,distinct_attribute_versions,source_files
230001,2,2,"List(products_20261008.json, products_20261007.json)"
230002,2,2,"List(products_20261008.json, products_20261007.json)"
230003,2,2,"List(products_20261008.json, products_20261007.json)"
230004,2,2,"List(products_20261008.json, products_20261007.json)"
230005,2,2,"List(products_20261008.json, products_20261007.json)"
230006,2,2,"List(products_20261008.json, products_20261007.json)"
